# Comparing Marketing Campaigns with Expected Utility (SOLUTION)

**Scenario.** You are the lead decision scientist at **Key & Quarter Hosts**
(fictional), a small short-term-rental management firm. Growth wants to launch
a campaign next quarter — *Premium Listing Push*, *Local Concierge Add-on*, or
*Hold* — and the payoff depends on whether the short-term-rental market is
Strong, Average, or Weak over the next three months.

This notebook works through all three decision rules — expected value, expected
CRRA utility, and minimax regret — and defends a choice between them.

## Setup

In [1]:
import numpy as np
import pandas as pd

DATA_PATH = "../comparing-marketing-campaigns-starter/data/airbnb_city_stats.csv"

## 1. Load the data

In [2]:
cities = pd.read_csv(DATA_PATH)
cities

,city,country,total_listings,median_price_usd,median_reviews_per_month,pct_entire_home
0,New York City,United States,37000,200,0.70,52
1,Los Angeles,United States,30000,240,0.50,68
2,Austin,United States,13000,240,0.55,72
3,Vancouver,Canada,5500,170,0.55,55
4,Mexico City,Mexico,25000,60,0.85,80
5,Buenos Aires,Argentina,15000,55,0.65,75
6,Rio de Janeiro,Brazil,22000,70,0.70,65
7,Cape Town,South Africa,22000,100,0.70,80
8,Paris,France,85000,110,0.60,85
9,London,United Kingdom,80000,120,0.50,60


## 2. Per-listing monthly revenue proxy

A simple revenue estimate per listing: median nightly price times median monthly
reviews per active listing (a standard occupancy proxy).

In [3]:
cities["revenue_proxy"] = (
    cities["median_price_usd"] * cities["median_reviews_per_month"]
)
cities[["city", "median_price_usd", "median_reviews_per_month",
        "revenue_proxy"]].sort_values("revenue_proxy").round(2)

,city,median_price_usd,median_reviews_per_month,revenue_proxy
5,Buenos Aires,55,0.65,35.75
16,Tokyo,95,0.40,38.00
10,Berlin,80,0.55,44.00
6,Rio de Janeiro,70,0.70,49.00
4,Mexico City,60,0.85,51.00
9,London,120,0.50,60.00
8,Paris,110,0.60,66.00
7,Cape Town,100,0.70,70.00
13,Madrid,110,0.75,82.50
3,Vancouver,170,0.55,93.50


## 3. Classify each city into Strong / Average / Weak by tertile

In [4]:
t33, t67 = cities["revenue_proxy"].quantile([1 / 3, 2 / 3])
print(f"33rd percentile cutoff: {t33:.2f}")
print(f"67th percentile cutoff: {t67:.2f}")


def classify(value: float) -> str:
    if value >= t67:
        return "Strong"
    if value >= t33:
        return "Average"
    return "Weak"


cities["environment"] = cities["revenue_proxy"].apply(classify)
cities[["city", "revenue_proxy", "environment"]]

33rd percentile cutoff: 64.00
67th percentile cutoff: 102.08


,city,revenue_proxy,environment
0,New York City,140.00,Strong
1,Los Angeles,120.00,Strong
2,Austin,132.00,Strong
3,Vancouver,93.50,Average
4,Mexico City,51.00,Weak
5,Buenos Aires,35.75,Weak
6,Rio de Janeiro,49.00,Weak
7,Cape Town,70.00,Average
8,Paris,66.00,Average
9,London,60.00,Weak


## 4. Empirical probability of each environment

In [5]:
counts = cities["environment"].value_counts()
p = (counts / counts.sum()).reindex(["Strong", "Average", "Weak"])
print(f"Sum: {p.sum():.3f}")
p.round(3)

Sum: 1.000


environment
Strong     0.333
Average    0.333
Weak       0.333
Name: count, dtype: float64

Since we split by tertile, each group has 6 cities — a third of the total.
That gives us equal probability across Strong, Average, and Weak: a simple,
clean baseline.

## 5. Payoff matrix

In [6]:
payoffs = pd.DataFrame(
    {
        "Strong":  {"Premium Listing Push": 10.0, "Local Concierge Add-on": 3.0, "Hold": 0.0},
        "Average": {"Premium Listing Push":  2.0, "Local Concierge Add-on": 1.5, "Hold": 0.0},
        "Weak":    {"Premium Listing Push": -6.0, "Local Concierge Add-on": 0.0, "Hold": 0.0},
    }
)
payoffs

,Strong,Average,Weak
Premium Listing Push,10.0,2.0,-6.0
Local Concierge Add-on,3.0,1.5,0.0
Hold,0.0,0.0,0.0


## 6. Expected value per option

In [7]:
ev = (payoffs * p).sum(axis=1)
print(f"EV-maximizing option: {ev.idxmax()}  (${ev.max():.2f}M)")
ev.round(3)

EV-maximizing option: Premium Listing Push  ($2.00M)


Premium Listing Push      2.0
Local Concierge Add-on    1.5
Hold                      0.0
dtype: float64

## 7. Expected CRRA utility per option

CRRA utility on (wealth_baseline + incremental profit), with γ = 2 and
W = $50M. The wealth baseline keeps utility well-defined and represents
the existing scale of the business. With γ > 1 the function is more sensitive
to losses than to equally-sized gains — which is what risk aversion means
in practice.

In [8]:
GAMMA = 2.0
WEALTH_M = 50.0


def crra_utility(profit_m, gamma: float = GAMMA, wealth_m: float = WEALTH_M):
    """CRRA utility evaluated at (wealth_m + profit_m). Vectorized."""
    x = wealth_m + np.asarray(profit_m, dtype=float)
    if abs(gamma - 1.0) < 1e-9:
        return np.log(x)
    return (np.power(x, 1.0 - gamma) - 1.0) / (1.0 - gamma)


utility_table = payoffs.map(crra_utility)
expected_utility = (utility_table * p).sum(axis=1)
ce_total_wealth = np.power(
    expected_utility * (1.0 - GAMMA) + 1.0, 1.0 / (1.0 - GAMMA)
)
certainty_equivalent = ce_total_wealth - WEALTH_M

print(f"Utility-maximizing option: {expected_utility.idxmax()}  "
      f"(CE = ${certainty_equivalent[expected_utility.idxmax()]:.2f}M)")
print("Expected utility per option:")
display(expected_utility.round(6))
print("Certainty-equivalent profit ($M) per option:")
certainty_equivalent.round(3)

Utility-maximizing option: Local Concierge Add-on  (CE = $1.47M)
Expected utility per option:


Premium Listing Push      0.980458
Local Concierge Add-on    0.980572
Hold                      0.980000
dtype: float64

Certainty-equivalent profit ($M) per option:


Premium Listing Push      1.173
Local Concierge Add-on    1.471
Hold                     -0.000
dtype: float64

Notice that *Premium Listing Push* has the highest **expected profit** but
*Local Concierge Add-on* has the highest **certainty-equivalent profit**.
A risk-averse decision-maker would trade away ~$0.5M of expected profit to
avoid the -$6M downside in a Weak environment.

## 8. Minimax regret per option

In [9]:
best_per_state = payoffs.max(axis=0)
regret = best_per_state - payoffs
max_regret = regret.max(axis=1)

print(f"Minimax-regret option: {max_regret.idxmin()}  "
      f"(max regret = ${max_regret.min():.2f}M)")
print("Regret matrix ($M):")
display(regret)
print("Max regret per option ($M):")
max_regret

Minimax-regret option: Premium Listing Push  (max regret = $6.00M)
Regret matrix ($M):


,Strong,Average,Weak
Premium Listing Push,0.0,0.0,6.0
Local Concierge Add-on,7.0,0.5,0.0
Hold,10.0,2.0,0.0


Max regret per option ($M):


Premium Listing Push       6.0
Local Concierge Add-on     7.0
Hold                      10.0
dtype: float64

## 9. Compare the three decision rules

In [10]:
comparison = pd.DataFrame(
    {
        "Selected option": [ev.idxmax(),
                            expected_utility.idxmax(),
                            max_regret.idxmin()],
        "Score": [f"EV = ${ev.max():.2f}M",
                  f"CE = ${certainty_equivalent[expected_utility.idxmax()]:.2f}M",
                  f"Max regret = ${max_regret.min():.2f}M"],
    },
    index=["EV-max", "Expected-utility-max (γ=2)", "Minimax regret"],
)
comparison

,Selected option,Score
EV-max,Premium Listing Push,EV = $2.00M
Expected-utility-max (γ=2),Local Concierge Add-on,CE = $1.47M
Minimax regret,Premium Listing Push,Max regret = $6.00M


**The three rules do not agree.** EV-max picks Premium Listing Push,
minimax-regret picks Premium Listing Push, but expected CRRA utility (γ = 2)
picks Local Concierge Add-on. The disagreement is real: Premium has the
higher expected profit but a heavier downside in a Weak market — its
certainty-equivalent profit ($1.17M) is lower than Local Concierge's
($1.47M) once the −$6M loss is priced in.

**The decision rule to lean on is expected CRRA utility.** With γ = 2,
the gap between Premium's EV ($2.0M) and Local Concierge's EV ($1.5M) does
not justify the −$6M downside in a Weak quarter for a firm of Key & Quarter Hosts's
scale. A risk-neutral decision-maker would go with EV-max instead.

Turning this into a full recommendation — with context and caveats — is a
separate step beyond this analysis.

## 10. Segmentation-aware option: Selective Push

Every option so far commits Key & Quarter Hosts to the *same campaign across its entire
portfolio*. But Key & Quarter Hosts can observe which markets are currently Strong, Average,
or Weak using the market data — and could target its spend accordingly.

**Selective Push**: run Premium Listing Push only in currently-Strong markets,
Local Concierge Add-on in Average markets, and Hold in Weak markets.

In [11]:
payoffs.loc["Selective Push"] = {"Strong": 6.0, "Average": 1.5, "Weak": -1.0}
payoffs

,Strong,Average,Weak
Premium Listing Push,10.0,2.0,-6.0
Local Concierge Add-on,3.0,1.5,0.0
Hold,0.0,0.0,0.0
Selective Push,6.0,1.5,-1.0


In [12]:
ev4           = (payoffs * p).sum(axis=1)
utility4      = payoffs.map(crra_utility)
exp_u4        = (utility4 * p).sum(axis=1)
ce4_wealth    = np.power(exp_u4 * (1.0 - GAMMA) + 1.0, 1.0 / (1.0 - GAMMA))
ce4           = ce4_wealth - WEALTH_M
best4         = payoffs.max(axis=0)
max_regret4   = (best4 - payoffs).max(axis=1)

comparison4 = pd.DataFrame(
    {
        "Selected option": [ev4.idxmax(),
                            exp_u4.idxmax(),
                            max_regret4.idxmin()],
        "Score": [f"EV = ${ev4.max():.2f}M",
                  f"CE = ${ce4[exp_u4.idxmax()]:.2f}M",
                  f"Max regret = ${max_regret4.min():.2f}M"],
    },
    index=["EV-max", "Expected-utility-max (γ=2)", "Minimax regret"],
)
comparison4

,Selected option,Score
EV-max,Selective Push,EV = $2.17M
Expected-utility-max (γ=2),Selective Push,CE = $2.01M
Minimax regret,Selective Push,Max regret = $4.00M


**All three rules now agree on Selective Push** (EV = $2.17M, CE = $2.01M,
max regret = $4.0M — better than every other option on every criterion).
The uniform options force Key & Quarter Hosts to either accept the full -$6M downside
(Premium) or cap the upside at $3M (Concierge). Selective Push avoids that
tradeoff: by running the expensive campaign only where conditions are strong
and limiting exposure elsewhere, it keeps most of Premium's upside while
cutting its worst-case loss by two-thirds.